# Thí nghiệm [09][TV2]: Gini Impurity vs Shannon Entropy (Information Gain)

Notebook này thực hiện so sánh đối chứng giữa hai tiêu chí phân chia trong cây quyết định:
- **Gini Impurity**: $G(S) = 1 - \sum_{i=1}^C p_i^2$
- **Shannon Entropy (Information Gain)**: $H(S) = -\sum_{i=1}^C p_i \log_2 p_i$

Mục tiêu là đánh giá độ chính xác (`accuracy`), `error_rate`, `macro_f1`, độ sâu cây (`tree_depth`), số lá (`leaf_count`) và thời gian huấn luyện trên cùng tập dữ liệu chuẩn **UCI Letter Recognition**.


In [ ]:
import sys
import time
from pathlib import Path
from zipfile import ZipFile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path("..").resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from decision_tree_lab2.config import (
    FIGURES_DIR,
    PROCESSED_DATA_DIR,
    RANDOM_STATE,
    RESULTS_DIR,
    TEST_SIZE,
)
from decision_tree_lab2.letter_data import LETTER_FEATURES, LETTER_TARGET

RANDOM_STATE = 42
TEST_SIZE = 0.20
pipeline_seconds = 0.0
data_loading_seconds = 0.0
training_seconds = 0.0
prediction_seconds = 0.0
model_compute_device = "CPU"
kaggle_working = Path("/kaggle/working")
outputs_zip = kaggle_working / "dt_letter_gini_vs_entropy__outputs.zip"
print(f"Project root: {PROJECT_ROOT}")


## 1. Tải dữ liệu chuẩn (Letter Recognition)


In [ ]:
t0 = time.perf_counter()
train_path = PROCESSED_DATA_DIR / "letter_recognition" / "train.csv"
test_path = PROCESSED_DATA_DIR / "letter_recognition" / "test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
data_loading_seconds = time.perf_counter() - t0

X_train = train.loc[:, list(LETTER_FEATURES)]
y_train = train[LETTER_TARGET]
X_test = test.loc[:, list(LETTER_FEATURES)]
y_test = test[LETTER_TARGET]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


## 2. Huấn luyện mô hình và so sánh Gini vs Entropy


In [ ]:
# !nvidia-smi
t_fit_start = time.perf_counter()
clf_gini = DecisionTreeClassifier(criterion="gini", random_state=RANDOM_STATE)
clf_gini.fit(X_train, y_train)
training_seconds = time.perf_counter() - t_fit_start

t_pred_start = time.perf_counter()
y_pred_gini = clf_gini.predict(X_test)
prediction_seconds = time.perf_counter() - t_pred_start
pipeline_seconds = data_loading_seconds + training_seconds + prediction_seconds

clf_entropy = DecisionTreeClassifier(criterion="entropy", random_state=RANDOM_STATE)
clf_entropy.fit(X_train, y_train)


## 3. Hiển thị kết quả tổng hợp


In [ ]:
summary_csv = RESULTS_DIR / "dt_letter_gini_vs_entropy__summary.csv"
if summary_csv.exists():
    df_res = pd.read_csv(summary_csv)
    display(df_res)
